#### Importing Libraries

In [1]:
# Importing libraries for data preparation
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

#### Loading the Dataset

In [2]:
# Loading the original product classification dataset
df = pd.read_csv("../data/raw/pricerunner_aggregate.csv")

# Displaying the first 10 records
df.head(10)

,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
5,6,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,6,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
6,7,apple iphone 8 plus 64 gb space grey,7,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
7,8,apple iphone 8 plus 64gb space grey,8,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
8,9,apple iphone 8 plus 64gb space grey,9,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
9,10,apple iphone 8 plus 64gb space grey,10,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


#### Inspecting the Dataset Structure

In [3]:
# Displaying the dataset dimensions
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Displaying column names and data types
print("\nDataset Information:")
df.info()

Number of rows: 35,311
Number of columns: 7

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Product ID       35311 non-null  int64
 1   Product Title    35311 non-null  str  
 2    Merchant ID     35311 non-null  int64
 3    Cluster ID      35311 non-null  int64
 4    Cluster Label   35311 non-null  str  
 5    Category ID     35311 non-null  int64
 6    Category Label  35311 non-null  str  
dtypes: int64(4), str(3)
memory usage: 4.9 MB


**Observations on Dataset Structure:**

- The dataset contains **35,311 product listings** and **7 columns**, consistent with the original data.
- All columns contain **35,311 non-null values**, indicating no missing data.
- The dataset includes **4 integer columns** and **3 text columns**, with leading spaces present in several column names.

#### Standardizing Column Names

In [4]:
# Standardizing column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Displaying the standardized column names
df.columns.tolist()

['Product_ID',
 'Product_Title',
 'Merchant_ID',
 'Cluster_ID',
 'Cluster_Label',
 'Category_ID',
 'Category_Label']

#### Identifying Duplicate Records

In [5]:
# Identifying exact duplicate records
exact_duplicates = df.duplicated().sum()

# Identifying duplicate records excluding the product identifier
duplicate_listings = df.duplicated(
    subset=["Product_Title", "Merchant_ID", "Cluster_ID", "Category_ID"]
).sum()

# Identifying repeated product titles
repeated_titles = df.duplicated(subset=["Product_Title"]).sum()

# Displaying duplicate counts
print(f"Exact duplicate records: {exact_duplicates:,}")
print(f"Duplicate listings excluding Product_ID: {duplicate_listings:,}")
print(f"Repeated product title occurrences: {repeated_titles:,}")

Exact duplicate records: 0
Duplicate listings excluding Product_ID: 0
Repeated product title occurrences: 4,318


**Observations on Duplicate Records:**

- No exact duplicate records or duplicate listings excluding `Product_ID` were identified.
- **4,318 repeated product title occurrences** were detected, reflecting identical titles across different product records.
- Repeated titles will be retained at this stage, with their cluster relationships considered during leakage-aware data splitting.

#### Inspecting Product Title Whitespace

In [6]:
# Identifying leading and trailing whitespace
leading_whitespace = df["Product_Title"].str.match(r"^\s").sum()
trailing_whitespace = df["Product_Title"].str.contains(r"\s$").sum()

# Identifying consecutive whitespace characters
repeated_whitespace = df["Product_Title"].str.contains(r"\s{2,}").sum()

# Identifying empty or whitespace-only titles
empty_titles = df["Product_Title"].str.strip().eq("").sum()

# Displaying whitespace-related findings
print(f"Titles with leading whitespace: {leading_whitespace:,}")
print(f"Titles with trailing whitespace: {trailing_whitespace:,}")
print(f"Titles with repeated internal whitespace: {repeated_whitespace:,}")
print(f"Empty or whitespace-only titles: {empty_titles:,}")

Titles with leading whitespace: 0
Titles with trailing whitespace: 0
Titles with repeated internal whitespace: 0
Empty or whitespace-only titles: 0


**Observations on Product Title Whitespace:**

- No product titles contain leading, trailing, or repeated internal whitespace.
- No empty or whitespace-only product titles were identified.
- No whitespace corrections are required, allowing us to preserve the original product descriptions.

#### Inspecting Product Title Capitalization

In [7]:
# Identifying titles containing uppercase letters
uppercase_titles = df["Product_Title"].str.contains(r"[A-Z]").sum()

# Identifying titles containing lowercase letters
lowercase_titles = df["Product_Title"].str.contains(r"[a-z]").sum()

# Calculating capitalization percentages
uppercase_percentage = uppercase_titles / len(df) * 100

# Displaying capitalization findings
print(f"Titles containing uppercase letters: {uppercase_titles:,}")
print(f"Titles containing lowercase letters: {lowercase_titles:,}")
print(f"Percentage containing uppercase letters: {uppercase_percentage:.2f}%")

Titles containing uppercase letters: 0
Titles containing lowercase letters: 35,311
Percentage containing uppercase letters: 0.00%


**Observations on Product Title Capitalization:**

- All **35,311 product titles** contain lowercase letters.
- No uppercase characters were identified across the dataset.
- Lowercase normalization is unnecessary, allowing the existing text to be retained without modification.

#### Identifying Conflicting Category Labels

In [8]:
# Identifying product titles associated with multiple category labels
title_category_counts = (
    df.groupby("Product_Title")["Category_Label"]
    .nunique()
)

# Selecting titles with conflicting category assignments
conflicting_titles = title_category_counts[
    title_category_counts > 1
].index

# Retrieving the affected records
conflicting_records = (
    df[df["Product_Title"].isin(conflicting_titles)]
    [["Product_ID", "Product_Title", "Cluster_ID", "Category_ID", "Category_Label"]]
    .sort_values("Product_Title")
    .reset_index(drop=True)
)

# Displaying the conflicting records
print(f"Conflicting product titles: {len(conflicting_titles):,}")
print(f"Affected records: {len(conflicting_records):,}")

display(conflicting_records)

Conflicting product titles: 2
Affected records: 4


,Product_ID,Product_Title,Cluster_ID,Category_ID,Category_Label
0,43365,blomberg kgm9550 fridge freezer,46085,2622,Fridge Freezers
1,47235,blomberg kgm9550 fridge freezer,47395,2623,Fridges
2,34378,bosch kur15a50gb integrated undercounter fridge,42819,2620,Washing Machines
3,43394,bosch kur15a50gb integrated undercounter fridge,46113,2623,Fridges


**Observations on Conflicting Category Labels:**

- **2 product titles**, affecting **4 records**, have inconsistent category assignments.
- `blomberg kgm9550 fridge freezer` is classified under both `Fridge Freezers` and `Fridges`.
- `bosch kur15a50gb integrated undercounter fridge` is classified under both `Washing Machines` and `Fridges`, indicating a likely labeling error.

#### Inspecting Clusters with Conflicting Category Assignments

In [9]:
# Identifying clusters associated with conflicting product titles
affected_cluster_ids = conflicting_records["Cluster_ID"].unique()

# Retrieving all listings belonging to the affected clusters
affected_clusters = (
    df[df["Cluster_ID"].isin(affected_cluster_ids)]
    [["Product_ID", "Product_Title", "Cluster_ID", "Cluster_Label", "Category_Label"]]
    .sort_values(["Cluster_ID", "Product_Title"])
    .reset_index(drop=True)
)

# Summarizing the affected clusters
cluster_summary = (
    affected_clusters.groupby(["Cluster_ID", "Category_Label"])
    .agg(
        Listing_Count=("Product_ID", "size"),
        Unique_Titles=("Product_Title", "nunique")
    )
    .reset_index()
)

print(f"Affected clusters: {len(affected_cluster_ids)}")
display(cluster_summary)

Affected clusters: 4


,Cluster_ID,Category_Label,Listing_Count,Unique_Titles
0,42819,Washing Machines,1,1
1,46085,Fridge Freezers,1,1
2,46113,Fridges,10,9
3,47395,Fridges,1,1


**Observations on Clusters with Conflicting Category Assignments:**

- Three affected clusters (`42819`, `46085`, and `47395`) contain only **1 listing each**, while cluster `46113` contains **10 listings**.
- The `Bosch KUR15A50GB` title appears in a singleton `Washing Machines` cluster and a larger `Fridges` cluster, supporting the possibility of an incorrect category assignment.
- The `Blomberg KGM9550` title appears in separate singleton clusters with different categories, requiring further inspection before resolving the inconsistency.

#### Inspecting Product Listings Within Affected Clusters

In [10]:
# Displaying product listings within the affected clusters
for cluster_id in sorted(affected_cluster_ids):
    cluster_records = affected_clusters[
        affected_clusters["Cluster_ID"] == cluster_id
    ]

    print(f"\nCluster ID: {cluster_id}")
    print(f"Category: {cluster_records['Category_Label'].iloc[0]}")
    print(f"Cluster Label: {cluster_records['Cluster_Label'].iloc[0]}")
    print(f"Number of Listings: {len(cluster_records)}")

    display(
        cluster_records[
            ["Product_ID", "Product_Title", "Cluster_Label"]
        ].reset_index(drop=True)
    )


Cluster ID: 42819
Category: Washing Machines
Cluster Label: Hotpoint WMUD962G
Number of Listings: 1


,Product_ID,Product_Title,Cluster_Label
0,34378,bosch kur15a50gb integrated undercounter fridge,Hotpoint WMUD962G



Cluster ID: 46085
Category: Fridge Freezers
Cluster Label: Blomberg KGM 9550 White
Number of Listings: 1


,Product_ID,Product_Title,Cluster_Label
0,43365,blomberg kgm9550 fridge freezer,Blomberg KGM 9550 White



Cluster ID: 46113
Category: Fridges
Cluster Label: Bosch KUR15A50GB Integrated
Number of Listings: 10


,Product_ID,Product_Title,Cluster_Label
0,43392,bosch kur15a50gb built in under larder fridge ...,Bosch KUR15A50GB Integrated
1,43400,bosch kur15a50gb built under fridge larder 141...,Bosch KUR15A50GB Integrated
2,43399,bosch kur15a50gb classixx integrated 82cm unde...,Bosch KUR15A50GB Integrated
3,43401,bosch kur15a50gb classixx integrated 82cm unde...,Bosch KUR15A50GB Integrated
4,43402,bosch kur15a50gb integrated built under fridge...,Bosch KUR15A50GB Integrated
5,43398,bosch kur15a50gb integrated under counter fridge,Bosch KUR15A50GB Integrated
6,43394,bosch kur15a50gb integrated undercounter fridge,Bosch KUR15A50GB Integrated
7,43395,bosch kur15a50gb/kur15a50gb,Bosch KUR15A50GB Integrated
8,43396,bosch serie 6 built under larder fridge in white,Bosch KUR15A50GB Integrated
9,43397,bosch serie 6 kur15a50gb classixx 60cm wide in...,Bosch KUR15A50GB Integrated



Cluster ID: 47395
Category: Fridges
Cluster Label: Blomberg KGM9550 Stainless Steel
Number of Listings: 1


,Product_ID,Product_Title,Cluster_Label
0,47235,blomberg kgm9550 fridge freezer,Blomberg KGM9550 Stainless Steel


**Observations on Product Listings Within Affected Clusters:**

- Cluster `42819` contains a Bosch refrigerator title incorrectly associated with a `Hotpoint WMUD962G` washing machine cluster, indicating a mismatched product listing.
- Cluster `46113` consistently represents the `Bosch KUR15A50GB` refrigerator, with 10 listings assigned to `Fridges`.
- Both Blomberg clusters represent variants of the `KGM9550` fridge freezer, suggesting that the `Fridges` assignment in cluster `47395` is inconsistent with the product type.

#### Resolving Conflicting Category Assignments

In [11]:
# Creating a copy before applying data quality corrections
df_clean = df.copy()

# Removing the mismatched Bosch product listing
df_clean = df_clean[
    df_clean["Product_ID"] != 34378
].copy()

# Correcting the Blomberg fridge freezer category assignment
blomberg_mask = df_clean["Product_ID"] == 47235

df_clean.loc[blomberg_mask, "Category_ID"] = 2622
df_clean.loc[blomberg_mask, "Category_Label"] = "Fridge Freezers"

# Resetting the dataframe index
df_clean.reset_index(drop=True, inplace=True)

# Verifying the corrected records
print(f"Original records: {len(df):,}")
print(f"Cleaned records: {len(df_clean):,}")
print(f"Records removed: {len(df) - len(df_clean):,}")

display(
    df_clean[
        df_clean["Product_ID"].isin([34378, 43365, 43394, 47235])
    ][["Product_ID", "Product_Title", "Category_ID", "Category_Label"]]
)

Original records: 35,311
Cleaned records: 35,310
Records removed: 1


C:\Users\user\AppData\Local\Temp\ipykernel_1856\3716315191.py:16: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.reset_index(drop=True, inplace=True)


,Product_ID,Product_Title,Category_ID,Category_Label
31699,43365,blomberg kgm9550 fridge freezer,2622,Fridge Freezers
31727,43394,bosch kur15a50gb integrated undercounter fridge,2623,Fridges
35195,47235,blomberg kgm9550 fridge freezer,2622,Fridge Freezers


#### Validating Category Label Consistency

In [12]:
# Checking whether identical product titles have conflicting category labels
remaining_conflicts = (
    df_clean.groupby("Product_Title")["Category_Label"]
    .nunique()
    .gt(1)
    .sum()
)

# Checking consistency between category IDs and category labels
category_id_conflicts = (
    df_clean.groupby("Category_ID")["Category_Label"]
    .nunique()
    .gt(1)
    .sum()
)

category_label_conflicts = (
    df_clean.groupby("Category_Label")["Category_ID"]
    .nunique()
    .gt(1)
    .sum()
)

# Displaying validation results
print(f"Titles with conflicting categories: {remaining_conflicts}")
print(f"Category IDs mapped to multiple labels: {category_id_conflicts}")
print(f"Category labels mapped to multiple IDs: {category_label_conflicts}")
print(f"Cleaned dataset dimensions: {df_clean.shape}")

Titles with conflicting categories: 0
Category IDs mapped to multiple labels: 0
Category labels mapped to multiple IDs: 0
Cleaned dataset dimensions: (35310, 7)


**Observations on Category Label Consistency:**

- No product titles have conflicting category assignments after cleaning.
- Category IDs and category labels maintain a consistent one-to-one mapping.
- The cleaned dataset contains **35,310 records and 7 columns**, with the identified labeling inconsistencies resolved.

#### Saving the Cleaned Dataset

In [13]:
# Creating the processed data directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Defining the cleaned dataset path
cleaned_data_path = processed_dir / "cleaned_products.csv"

# Saving the cleaned dataset
df_clean.to_csv(cleaned_data_path, index=False)

# Verifying the saved dataset
saved_df = pd.read_csv(cleaned_data_path)

print(f"Saved dataset: {cleaned_data_path}")
print(f"Saved dataset dimensions: {saved_df.shape}")
print(f"Data matches cleaned dataset: {saved_df.equals(df_clean)}")

Saved dataset: ..\data\processed\cleaned_products.csv
Saved dataset dimensions: (35310, 7)
Data matches cleaned dataset: True


#### Identifying Predictor and Target Variables

In [14]:
# Defining the predictor and target columns
feature_column = "Product_Title"
target_column = "Category_Label"

# Identifying columns required for leakage-aware splitting
grouping_columns = ["Cluster_ID", "Product_Title"]

# Separating predictor and target variables
X = df_clean[[feature_column]].copy()
y = df_clean[target_column].copy()

# Displaying dataset dimensions
print(f"Predictor dimensions: {X.shape}")
print(f"Target dimensions: {y.shape}")
print(f"Number of target classes: {y.nunique()}")

# Displaying the target class distribution
display(y.value_counts().rename_axis("Category").reset_index(name="Count"))

Predictor dimensions: (35310, 1)
Target dimensions: (35310,)
Number of target classes: 10


,Category,Count
0,Fridge Freezers,5502
1,Mobile Phones,4081
2,Washing Machines,4043
3,CPUs,3862
4,Fridges,3583
5,TVs,3564
6,Dishwashers,3424
7,Digital Cameras,2697
8,Microwaves,2342
9,Freezers,2212


**Observations on Predictor and Target Variables:**

- The dataset contains **35,310 product titles** as predictors and **10 target categories** for multiclass classification.
- `Fridge Freezers` remains the largest category (**5,502 records**), while `Freezers` is the smallest (**2,212 records**).
- Product identifiers, merchant identifiers, cluster descriptions, and category IDs are excluded from predictive features to maintain a text-based classification approach.

#### Constructing Leakage-Aware Product Groups

In [15]:
# Initializing disjoint-set structures for product clusters
cluster_ids = df_clean["Cluster_ID"].unique()
parent = {cluster_id: cluster_id for cluster_id in cluster_ids}
rank = {cluster_id: 0 for cluster_id in cluster_ids}

# Defining a function to identify the root of each connected group
def find(cluster_id):
    if parent[cluster_id] != cluster_id:
        parent[cluster_id] = find(parent[cluster_id])
    return parent[cluster_id]

# Defining a function to merge connected clusters
def union(cluster_a, cluster_b):
    root_a = find(cluster_a)
    root_b = find(cluster_b)

    if root_a == root_b:
        return

    if rank[root_a] < rank[root_b]:
        root_a, root_b = root_b, root_a

    parent[root_b] = root_a

    if rank[root_a] == rank[root_b]:
        rank[root_a] += 1

# Connecting clusters that share identical product titles
for _, cluster_group in df_clean.groupby("Product_Title")["Cluster_ID"]:
    unique_clusters = cluster_group.unique()

    for cluster_id in unique_clusters[1:]:
        union(unique_clusters[0], cluster_id)

# Assigning a connected group identifier to each record
df_clean["Split_Group"] = df_clean["Cluster_ID"].map(find)

# Summarizing the connected groups
group_sizes = df_clean.groupby("Split_Group").size()

print(f"Original product clusters: {df_clean['Cluster_ID'].nunique():,}")
print(f"Connected product groups: {df_clean['Split_Group'].nunique():,}")
print(f"Largest connected group: {group_sizes.max():,} records")
print(f"Median connected group size: {group_sizes.median():.0f} records")

Original product clusters: 13,232
Connected product groups: 11,972
Largest connected group: 868 records
Median connected group size: 2 records


**Observations on Leakage-Aware Product Groups:**

- The cleaned dataset contains **13,232 original product clusters**, consolidated into **11,972 connected groups** based on shared clusters and identical product titles.
- The median connected group contains **2 records**, indicating that many groups remain relatively small.
- The largest connected group contains **868 records**, highlighting the need to examine group composition before performing a leakage-aware train/test split.

#### Inspecting Connected Group Category Consistency

In [16]:
# Calculating the number of distinct categories within each connected group
group_category_counts = (
    df_clean.groupby("Split_Group")["Category_Label"]
    .nunique()
)

# Identifying connected groups containing multiple categories
mixed_category_groups = group_category_counts[
    group_category_counts > 1
]

# Summarizing category consistency
print(f"Total connected groups: {len(group_category_counts):,}")
print(f"Single-category groups: {(group_category_counts == 1).sum():,}")
print(f"Mixed-category groups: {len(mixed_category_groups):,}")

# Inspecting the largest connected groups
largest_groups = (
    df_clean.groupby("Split_Group")
    .agg(
        Listing_Count=("Product_ID", "size"),
        Cluster_Count=("Cluster_ID", "nunique"),
        Unique_Titles=("Product_Title", "nunique"),
        Category_Count=("Category_Label", "nunique")
    )
    .sort_values("Listing_Count", ascending=False)
    .head(10)
)

display(largest_groups)

Total connected groups: 11,972
Single-category groups: 11,972
Mixed-category groups: 0


,Listing_Count,Cluster_Count,Unique_Titles,Category_Count
Split_Group,,,,
41985,868,140,619,1
44038,492,74,373,1
40466,362,63,274,1
41888,261,36,201,1
44051,235,36,171,1
44062,170,28,130,1
46116,102,14,74,1
4421,100,5,92,1
4419,89,4,76,1


**Observations on Connected Group Category Consistency:**

- All **11,972 connected product groups** contain listings from a single category, with no mixed-category groups identified.
- The largest group contains **868 listings**, spanning **140 clusters** and **619 unique product titles**.
- The presence of several large connected groups highlights the importance of preserving group boundaries while maintaining representative category distributions.

#### Preparing Connected Groups for Stratified Splitting

In [17]:
# Creating a summary of connected product groups
group_summary = (
    df_clean.groupby("Split_Group")
    .agg(
        Category_Label=("Category_Label", "first"),
        Listing_Count=("Product_ID", "size"),
        Cluster_Count=("Cluster_ID", "nunique")
    )
    .reset_index()
)

# Summarizing group availability across target categories
category_group_summary = (
    group_summary.groupby("Category_Label")
    .agg(
        Group_Count=("Split_Group", "nunique"),
        Total_Listings=("Listing_Count", "sum"),
        Largest_Group=("Listing_Count", "max")
    )
    .sort_values("Total_Listings", ascending=False)
    .reset_index()
)

# Calculating each category's average connected group size
category_group_summary["Average_Group_Size"] = (
    category_group_summary["Total_Listings"]
    / category_group_summary["Group_Count"]
).round(2)

# Displaying the group-level summary
display(category_group_summary)

print(f"Total connected groups: {len(group_summary):,}")
print(f"Total product listings: {group_summary['Listing_Count'].sum():,}")

,Category_Label,Group_Count,Total_Listings,Largest_Group,Average_Group_Size
0,Fridge Freezers,1771,5502,492,3.11
1,Mobile Phones,1667,4081,45,2.45
2,Washing Machines,1001,4043,868,4.04
3,CPUs,1729,3862,14,2.23
4,Fridges,1269,3583,102,2.82
5,TVs,1222,3564,100,2.92
6,Dishwashers,1249,3424,362,2.74
7,Digital Cameras,701,2697,27,3.85
8,Microwaves,693,2342,47,3.38
9,Freezers,670,2212,87,3.30


Total connected groups: 11,972
Total product listings: 35,310


**Observations on Group-Level Stratification:**

- `Fridge Freezers` have the most connected groups (**1,771**), while `Freezers` have the fewest (**670**).
- `Washing Machines` contain the largest connected group (**868 listings**) and have the highest average group size (**4.04**) among the categories.
- All categories contain hundreds of independent groups, supporting stratified splitting while requiring additional checks for record-level balance.

#### Creating Stratified Training and Testing Groups

In [19]:
# Splitting connected groups while stratifying by product category
train_groups, test_groups = train_test_split(
    group_summary,
    test_size=0.20,
    random_state=42,
    stratify=group_summary["Category_Label"]
)

# Retrieving product records associated with each partition
train_df = df_clean[
    df_clean["Split_Group"].isin(train_groups["Split_Group"])
].copy()

test_df = df_clean[
    df_clean["Split_Group"].isin(test_groups["Split_Group"])
].copy()

# Resetting dataset indices
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Displaying split dimensions
print(f"Training records: {len(train_df):,}")
print(f"Testing records: {len(test_df):,}")
print(f"Training percentage: {len(train_df) / len(df_clean) * 100:.2f}%")
print(f"Testing percentage: {len(test_df) / len(df_clean) * 100:.2f}%")

# Displaying group counts
print(f"\nTraining groups: {train_df['Split_Group'].nunique():,}")
print(f"Testing groups: {test_df['Split_Group'].nunique():,}")

Training records: 28,919
Testing records: 6,391
Training percentage: 81.90%
Testing percentage: 18.10%

Training groups: 9,577
Testing groups: 2,395


**Observations on Stratified Training and Testing Groups:**

- The training dataset contains **28,919 records (81.90%)**, while the testing dataset contains **6,391 records (18.10%)**.
- The split assigns **9,577 connected groups** to training and **2,395 groups** to testing, preserving group boundaries.
- The deviation from the intended 80/20 record-level ratio reflects differences in connected group sizes and requires further evaluation before finalizing the split.

#### Evaluating Category Distribution Across Training and Testing Sets

In [20]:
# Calculating category distributions across datasets
category_distribution = pd.DataFrame({
    "Original (%)": df_clean["Category_Label"].value_counts(normalize=True) * 100,
    "Training (%)": train_df["Category_Label"].value_counts(normalize=True) * 100,
    "Testing (%)": test_df["Category_Label"].value_counts(normalize=True) * 100
})

# Calculating absolute differences between training and testing proportions
category_distribution["Difference (pp)"] = (
    category_distribution["Training (%)"]
    - category_distribution["Testing (%)"]
).abs()

# Sorting by original category prevalence
category_distribution = category_distribution.sort_values(
    "Original (%)", ascending=False
).round(2)

# Displaying category distribution comparison
display(category_distribution)

print(
    f"Maximum category distribution difference: "
    f"{category_distribution['Difference (pp)'].max():.2f} percentage points"
)

,Original (%),Training (%),Testing (%),Difference (pp)
Category_Label,,,,
Fridge Freezers,15.58,15.79,14.63,1.16
Mobile Phones,11.56,11.71,10.89,0.81
Washing Machines,11.45,11.62,10.69,0.93
CPUs,10.94,10.79,11.63,0.84
Fridges,10.15,10.04,10.64,0.60
TVs,10.09,10.33,9.03,1.30
Dishwashers,9.70,9.85,9.00,0.85
Digital Cameras,7.64,7.34,9.00,1.66
Microwaves,6.63,6.49,7.28,0.79


Maximum category distribution difference: 1.66 percentage points


**Observations on Category Distribution Across Training and Testing Sets:**

- All **10 product categories** are represented in both training and testing datasets, with proportions generally close to the original distribution.
- `Digital Cameras` show the largest difference (**1.66 percentage points**), followed by `TVs` (1.30) and `Freezers` (1.18).
- The relatively small category distribution differences suggest that the group-aware split provides acceptable class representation for multiclass model evaluation.

#### Validating Training and Testing Data Separation

In [21]:
# Checking for overlapping connected groups
overlapping_groups = set(train_df["Split_Group"]) & set(test_df["Split_Group"])

# Checking for overlapping product clusters
overlapping_clusters = set(train_df["Cluster_ID"]) & set(test_df["Cluster_ID"])

# Checking for identical product titles across partitions
overlapping_titles = set(train_df["Product_Title"]) & set(test_df["Product_Title"])

# Checking for overlapping product identifiers
overlapping_products = set(train_df["Product_ID"]) & set(test_df["Product_ID"])

# Verifying that all cleaned records are accounted for
total_split_records = len(train_df) + len(test_df)

print(f"Overlapping connected groups: {len(overlapping_groups)}")
print(f"Overlapping product clusters: {len(overlapping_clusters)}")
print(f"Overlapping product titles: {len(overlapping_titles)}")
print(f"Overlapping product IDs: {len(overlapping_products)}")
print(f"All records accounted for: {total_split_records == len(df_clean)}")

# Enforcing leakage validation
assert len(overlapping_groups) == 0
assert len(overlapping_clusters) == 0
assert len(overlapping_titles) == 0
assert len(overlapping_products) == 0
assert total_split_records == len(df_clean)

print("\nAll train/test separation checks passed.")

Overlapping connected groups: 0
Overlapping product clusters: 0
Overlapping product titles: 0
Overlapping product IDs: 0
All records accounted for: True

All train/test separation checks passed.


**Observations on Training and Testing Data Separation:**

- No overlapping connected groups, product clusters, product titles, or product identifiers were found between training and testing datasets.
- All **35,310 cleaned records** are accounted for, with no records lost during splitting.
- The validated split prevents exact-title and cluster overlap, providing a reliable foundation for independent model evaluation.

#### Saving the Training and Testing Datasets

In [22]:
# Defining output paths for training and testing datasets
train_path = processed_dir / "train.csv"
test_path = processed_dir / "test.csv"

# Saving the validated datasets
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

# Reloading the saved datasets for verification
saved_train = pd.read_csv(train_path)
saved_test = pd.read_csv(test_path)

# Verifying saved data integrity
assert saved_train.equals(train_df)
assert saved_test.equals(test_df)

# Displaying saved dataset information
print(f"Training dataset saved: {train_path}")
print(f"Testing dataset saved: {test_path}")
print(f"Training dimensions: {saved_train.shape}")
print(f"Testing dimensions: {saved_test.shape}")
print("Saved datasets verified successfully.")

Training dataset saved: ..\data\processed\train.csv
Testing dataset saved: ..\data\processed\test.csv
Training dimensions: (28919, 8)
Testing dimensions: (6391, 8)
Saved datasets verified successfully.
